In [ ]:
import scanpy as sc

from src.models.destvi import DestVI
from src.models.condscvi import CondSCVI

# 完全按照原始DestVI的使用方式
# sc_adata = sc.read_h5ad("../data/simulation_Mouse_Kidney_MERFISH/sc_reference_TMS_droplets/tabula-muris-senis-droplet-processed-official-annotations-Kidney.h5ad")
# st_adata = sc.read_h5ad("../data/simulation_Mouse_Kidney_MERFISH/MK_simulated_ST.h5ad")
sc_adata = sc.read_h5ad("../scvi-tools-DestVI/data/sc_lymph_node_preprocessed.h5ad")
st_adata = sc.read_h5ad("../scvi-tools-DestVI/data/st_lymph_node_preprocessed.h5ad")
print(sc_adata)
print(st_adata)

In [ ]:
# 训练CondSCVI
CondSCVI.setup_anndata(sc_adata, labels_key="broad_cell_types")
sc_model = CondSCVI(sc_adata)
sc_model.train(max_epochs=400)

In [ ]:
# 使用独立的DestVI
DestVI.setup_anndata(st_adata, layer='counts')
spatial_model = DestVI.from_rna_model(st_adata, sc_model)
spatial_model.train(max_epochs=2000)

In [ ]:
# 获取结果
proportions = spatial_model.get_proportions()
gamma = spatial_model.get_gamma()

print("DestVI training completed!")
print(f"Proportions shape: {proportions.shape}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 保存 proportions 为 CSV 文件
cell_type_names = sc_adata.obs["broad_cell_types"].unique().tolist()
proportions_df = pd.DataFrame(proportions, columns=cell_type_names)
proportions_df.to_csv("destvi_predicted_proportions.csv", index=True)
print("Proportions saved to destvi_predicted_proportions.csv")

In [ ]:
# 可视化部分细胞类型比例（假设 st_adata.obsm["spatial"] 有空间坐标）
cell_types_to_plot = proportions_df.columns[:3]  # 选前3个细胞类型
coords = st_adata.obsm.get("spatial", None)
if coords is not None:
    for ct in cell_types_to_plot:
        plt.figure(figsize=(6, 5))
        plt.scatter(coords[:, 0], coords[:, 1], c=proportions_df[ct], cmap="Reds", s=10)
        plt.colorbar(label=f"{ct} proportion")
        plt.title(f"Spatial distribution of {ct}")
        plt.xlabel("X")
        plt.ylabel("Y")
        plt.tight_layout()
        plt.savefig(f"spatial_{ct}_proportion.png")
        plt.close()
    print("Spatial proportion plots saved as PNG.")
else:
    print("No spatial coordinates found in st_adata.obsm['spatial'], skipping spatial plots.")